## Notebook Experimentos Modelos MLflow

Objetivo:
  - Probar diferentes algoritmos y configuraciones para predecir Gasto_Total_COP.
  - Registrar parámetros y métricas en MLflow.
  - Identificar el mejor modelo candidato para producción.

### Librerias

In [3]:
import sys
print(sys.executable)  # solo para verificar que sigue siendo Python312

# Instalar mlflow en ESTE intérprete
!"{sys.executable}" -m pip install mlflow


C:\Users\Lenovo\AppData\Local\Programs\Python\Python312\python.exe
  Using cached mlflow-3.6.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.6.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_tracing-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached alembic-1.17.1-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.3-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.5.4-py3-none-any.whl.metadata (4.6 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.73.0-py3-none-any.whl.metadata (40 kB)
  Using cached fastapi-0.121.1-py3-none-any.whl.metadata (28 kB)
  Using cached opentelemetry_api-1.38.0-py3-none-any.whl.met

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.66.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.0 which is incompatible.
google-api-core 2.24.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.19.5, but you have protobuf 6.33.0 which is incompatible.
google-cloud-speech 2.29.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.0 which is incompatible.
grpcio-status 1.68.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.0 which is incompatible.
proto-plus 1.25.0 requires protobuf<6.0.0dev,>=3.19.0, but you have protobuf 6.33.0 which is incompatible.
streamlit 1.39.0 requires protobuf<6,>=3.20, but you have protobuf 6

In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from math import sqrt
from sklearn.ensemble import GradientBoostingRegressor

import mlflow
import mlflow.sklearn

### Cargue de Datos

In [15]:
# Cargar dataset procesado
ruta = "../data/processed/viajeros_2023_gasto_cop.csv"
df = pd.read_csv(ruta)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16616\2263007964.py:3: DtypeWarning: Columns (22,32,34,35,36,37,57,61,82,83,89,91,93,94,99,104,112,115,116,117,118,119,120,123,124,125) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)


### Definición de variables

In [18]:
# 2. Definir variables predictoras y objetivo
target = "Gasto_Total_COP"

# Ejemplo de selección (luego refinamos):
features_cat = [
    "P_102",      # país/origen
    "P_103",      # motivo viaje
    "P_107"       # tipo alojamiento
]
features_num = [
    "P_106A"      # noches de alojamiento
]

df_model = df[features_cat + features_num + [target]].dropna()

X = df_model[features_cat + features_num]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Procesamiento

In [19]:
# 3. Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
        # En este caso no escalamos numéricas; lo podemos agregar si se requiere.
        ("num", "passthrough", features_num)
    ]
)


In [26]:
def entrenar_y_loguear(nombre_experimento, modelo_sklearn, params=None):
    """
    Entrena un pipeline (preprocesamiento + modelo),
    evalúa y registra resultados en MLflow.
    """
    mlflow.set_experiment(nombre_experimento)

    with mlflow.start_run():
        pipe = Pipeline(steps=[
            ("pre", preprocessor),
            ("model", modelo_sklearn)
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        rmse = sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        if params:
            for k, v in params.items():
                mlflow.log_param(k, v)

        mlflow.log_metric("MSE", mse)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("R2", r2)

        mlflow.sklearn.log_model(pipe, "model")

        if params:
            print(f"Parámetros: {params}")
        print(
            f"Modelo: {modelo_sklearn.__class__.__name__} | "
            f"RMSE={rmse:.2f} | MAE={mae:.2f} | R2={r2:.3f}"
        )


### Modelos

### A. Regresión lineal (baseline)

In [27]:
# Regresión lineal (baseline)
entrenar_y_loguear(
    nombre_experimento="gasto_turistico_regresion_lineal",
    modelo_sklearn=LinearRegression(),
    params={"modelo": "LinearRegression"}
)


2025/11/09 23:40:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:40:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'modelo': 'LinearRegression'}
Modelo: LinearRegression | RMSE=7958030.33 | MAE=2304097.25 | R2=0.141


### B. Random Forest

In [28]:

# Random Forest con hiperparámetros distintos
for n in [100, 300]:
    for depth in [5, 10, None]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=42,
            n_jobs=-1
        )
        entrenar_y_loguear(
            nombre_experimento="gasto_turistico_random_forest",
            modelo_sklearn=rf,
            params={"n_estimators": n, "max_depth": depth}
        )


2025/11/09 23:40:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:40:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'n_estimators': 100, 'max_depth': 5}
Modelo: RandomForestRegressor | RMSE=7976917.92 | MAE=2203533.12 | R2=0.137


2025/11/09 23:40:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'n_estimators': 100, 'max_depth': 10}
Modelo: RandomForestRegressor | RMSE=8259776.33 | MAE=2285715.85 | R2=0.074


2025/11/09 23:41:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

Parámetros: {'n_estimators': 300, 'max_depth': 5}
Modelo: RandomForestRegressor | RMSE=7978822.16 | MAE=2207990.90 | R2=0.136


2025/11/09 23:41:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'n_estimators': 300, 'max_depth': 10}
Modelo: RandomForestRegressor | RMSE=8247393.51 | MAE=2285676.45 | R2=0.077


2025/11/09 23:41:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'n_estimators': 300, 'max_depth': None}
Modelo: RandomForestRegressor | RMSE=8253255.18 | MAE=2302537.49 | R2=0.076


### C. Gradient Boosting

In [29]:
for lr in [0.05, 0.1]:
    for n in [100, 200]:
        gbr = GradientBoostingRegressor(
            learning_rate=lr,
            n_estimators=n,
            random_state=42
        )
        entrenar_y_loguear(
            nombre_experimento="gasto_turistico_gradient_boosting",
            modelo_sklearn=gbr,
            params={"learning_rate": lr, "n_estimators": n}
        )


2025/11/09 23:41:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'learning_rate': 0.05, 'n_estimators': 100}
Modelo: GradientBoostingRegressor | RMSE=7904817.97 | MAE=2200991.21 | R2=0.152


2025/11/09 23:41:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'learning_rate': 0.05, 'n_estimators': 200}
Modelo: GradientBoostingRegressor | RMSE=8000806.41 | MAE=2254743.73 | R2=0.131


2025/11/09 23:41:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'learning_rate': 0.1, 'n_estimators': 100}
Modelo: GradientBoostingRegressor | RMSE=8022460.08 | MAE=2259512.42 | R2=0.127


2025/11/09 23:41:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 23:41:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Parámetros: {'learning_rate': 0.1, 'n_estimators': 200}
Modelo: GradientBoostingRegressor | RMSE=8065023.96 | MAE=2289917.15 | R2=0.117
